# Transformer


## Tokenization


In [ ]:
import numpy as np
from typing import List, Dict

class SimpleTokenizer:
    """
    A word-level tokenizer with special tokens.
    """

    def __init__(self):
        self.word_to_id: Dict[str, int] = {}
        self.id_to_word: Dict[int, str] = {}
        self.vocab_size = 0

        # Special tokens
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"

    def build_vocab(self, texts: List[str]) -> None:
        """
        Build vocabulary from a list of texts.
        Add special tokens first, then unique words.
        """
        for i,special in enumerate(["<PAD>","<UNK>","<BOS>","<EOS>"]):
            self.word_to_id[special] = i
            self.id_to_word[i] = special
            self.vocab_size += 1

        self.unique_words = sorted(set(word for text in texts for word in text.split()))
        for i,word in enumerate(self.unique_words):
            self.word_to_id[word.lower()] = i+4
            self.id_to_word[i+4] = word.lower()
            self.vocab_size += 1

    def encode(self, text: str) -> List[int]:
        """
        Convert text to list of token IDs.
        Use UNK for unknown words.
        """
        encoded = sorted(text.lower().split())
        return [self.word_to_id[enc] if enc in self.unique_words else self.word_to_id['<UNK>'] for enc in encoded]

    def decode(self, ids: List[int]) -> str:
        """
        Convert list of token IDs back to text.
        """
        decoded = ""
        print(ids)
        for id in ids:
            if self.id_to_word.get(id):
                decoded += self.id_to_word[id] + " "
            else:
                decoded += self.id_to_word[1] + " "
        return decoded.strip()

## Embeddings


In [1]:
import torch
import torch.nn as nn
import math

def create_embedding_layer(vocab_size: int, d_model: int) -> nn.Embedding:
    """
    Create an embedding layer.
    """
    return nn.Embedding(vocab_size,d_model)

def embed_tokens(embedding: nn.Embedding, tokens: torch.Tensor, d_model: int) -> torch.Tensor:
    """
    Convert token indices to scaled embeddings.
    """
    return embedding(tokens) * math.sqrt(d_model)

d = 32
emb = create_embedding_layer(int(d/2),d)
print(emb)
input = torch.LongTensor([[1, 2, 4, 5], [4, 3, 2, 9]])

print(embed_tokens(emb,input,d)[0,0])


input2 = torch.LongTensor([1])
print(embed_tokens(emb,input2,d)[0])

Embedding(16, 32)
tensor([-11.9463,   0.5838,  17.7782,  -2.7316,   2.2686,   3.5270,  -6.3466,
          4.1815,   2.3239,  -2.6793,   0.9449,   1.7787,   4.3670,  -0.6938,
         -0.4102,  -1.0505,  -6.4134,  -4.2080,   1.8470,   2.5783,   1.9728,
          4.2292,   7.7289,   4.9964,  -7.3568,  -9.2918,  -1.9854,  12.8763,
          5.0758,   1.4764,   3.3546,  12.3423], grad_fn=<SelectBackward0>)
tensor([-11.9463,   0.5838,  17.7782,  -2.7316,   2.2686,   3.5270,  -6.3466,
          4.1815,   2.3239,  -2.6793,   0.9449,   1.7787,   4.3670,  -0.6938,
         -0.4102,  -1.0505,  -6.4134,  -4.2080,   1.8470,   2.5783,   1.9728,
          4.2292,   7.7289,   4.9964,  -7.3568,  -9.2918,  -1.9854,  12.8763,
          5.0758,   1.4764,   3.3546,  12.3423], grad_fn=<SelectBackward0>)


## Positional Encoding


In [94]:
import numpy as np

def positional_encoding(seq_length: int, d_model: int) -> np.ndarray:
    """
    Generate sinusoidal positional encodings.
    """
    res = np.zeros((seq_length,d_model),dtype=np.float32)
    pos = np.arange(seq_length).reshape(-1, 1)
    
    i = np.arange(0,d_model//2,1)
    div = np.exp((2*i/d_model)*-np.log(10000))
    
    res[:,0::2] = np.sin(pos*div)
    res[:,1::2] = np.cos(pos*div)
    
    return res.round(4)
    

positional_encoding(4,6)


array([[ 0.    ,  1.    ,  0.    ,  1.    ,  0.    ,  1.    ],
       [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.    ],
       [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.    ],
       [ 0.1411, -0.99  ,  0.1388,  0.9903,  0.0065,  1.    ]],
      dtype=float32)

$
Q \in \mathbb{R}^{n \times d_k} \quad \text{(queries)}
$

$
K \in \mathbb{R}^{m \times d_k} \quad \text{(keys, must match query dimension)}
$

$
V \in \mathbb{R}^{m \times d_v} \quad \text{(values, rows must match keys)}
$

$
S \in \mathbb{R}^{n \times m} \quad \text{(score matrix)}
$

$
W \in \mathbb{R}^{n \times m} \quad \text{(attention weights, rows sum to 1)}
$

$
O \in \mathbb{R}^{n \times d_v} \quad \text{(output, one vector per query)}
$


## Attention


In [1]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """
    Compute scaled dot-product attention.
    """
    S = Q @ K.transpose(-2,-1) # n×m
    S_sq = S/math.sqrt(K.shape[-1]) # n×m
    
    W = F.softmax(S_sq,dim=-1)
    O = W @ V # m×d_v

    return O

Q = torch.FloatTensor([[[1,0],[0,1]]])
K = torch.FloatTensor([[[1,0],[0,1]]])
V = torch.FloatTensor([[[1,2],[3,4]]])

scaled_dot_product_attention(Q,K,V)

tensor([[[1.6605, 2.6605],
         [2.3395, 3.3395]]])

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def multi_head_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                         W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray,
                         W_o: np.ndarray, num_heads: int) -> np.ndarray:
    """
    Compute multi-head attention.
    """
    
    heads = []
    d_k = K.shape[-1]//num_heads
    
    for h in range(num_heads):
        Qi = (Q * W_q[h]).reshape(Q.shape[0],Q.shape[1],num_heads,d_k)\
            .reshape(Q.shape[0],num_heads,Q.shape[1],d_k)
        Ki = (K * W_k[h]).reshape(K.shape[0],K.shape[1],num_heads,d_k)\
            .reshape(Q.shape[0],num_heads,Q.shape[1],d_k)
        Vi = (V * W_v[h]).reshape(V.shape[0],V.shape[1],num_heads,d_k)\
            .reshape(Q.shape[0],num_heads,Q.shape[1],d_k)
        
        S = Qi @ Ki.transpose(0,1,3,2) # n×m
        S_sq = S/math.sqrt(K.shape[-1]) # n×m
        
        W = softmax(S_sq,axis=-1)
        O = W @ Vi # m×d_v

        heads.append(O.transpose(1,2))
    heads = np.array(heads).reshape(1,2,3,4) @ W_o
    return heads.sum(axis=-1)


Q = np.array([[[0.2484,-0.0691,0.3238,0.7615],
                [-0.1171,-0.1171,0.7896,0.3837],
                [-0.2347,0.2713,-0.2317,-0.2329]]])

K = np.array([[[0.121,-0.9566,-0.8625,-0.2811],
                [-0.5064,0.1571,-0.454,-0.7062],
                [0.7328,-0.1129,0.0338,-0.7124]]])

V = np.array([[[-0.2722,0.0555,-0.5755,0.1878],
                [-0.3003,-0.1458,-0.3009,0.9261],
                [-0.0067,-0.5289,0.4113,-0.6104]]])

W_q = np.array([[0.0627,-0.5879,-0.3985,0.0591],
                [0.2215,0.0514,-0.0347,-0.0903],
                [-0.4436,-0.216,-0.1382,0.3171],
                [0.1031,-0.5289,0.0972,-0.1155]])

W_k = np.array([[-0.2031,0.1835,0.3093,0.2794],
                [-0.2518,-0.0928,0.0994,0.2927],
                [-0.1438,-0.0557,-0.3319,-0.3589],
                [0.2438,0.4069,-0.0216,0.3011]])

W_v = np.array([[0.1085,-0.1935,0.1084,0.4614],
                [-0.0107,0.4694,-0.7859,0.2466],
                [0.0261,-0.0897,0.0275,-0.5963],
                [-0.0659,0.1071,0.4434,-0.1555]])

W_o = np.array([[-0.2425,-0.1505,0.2746,0.0986],
                [-0.1589,0.154,0.0291,0.2906],
                [-0.2106,-0.0983,-0.1176,-0.4391],
                [0.0888,0.0783,0.0015,-0.0704]])

num_heads = 2
multi_head_attention(Q,K,V,W_q,W_k,W_v,W_o,num_heads)

ValueError: axes don't match array

In [78]:
W_o.shape

(4, 4)